# XTTS Embedding Extraction — Emilia Dataset Sample

Loads one audio sample from the Emilia dataset (streamed, no full download), then extracts the two XTTS conditioning embeddings:
- `gpt_cond_latent` — style/prosody latent fed into the GPT
- `speaker_embedding` — d-vector fed into the HiFiGAN decoder

In [1]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [2]:
from datasets import load_dataset

# Stream one English sample — no full download
ds = load_dataset(
    "amphion/Emilia-Dataset",
    split="train",
    streaming=True,
)
sample = next(iter(ds))

print("Keys:", list(sample.keys()))
print("Text:", sample.get("text", sample.get("json", {}).get("text", "—")))

audio = sample["mp3"]  # dict with 'array' and 'sampling_rate'
print(
    f"Sample rate: {audio['sampling_rate']} Hz, length: {len(audio['array'])} samples"
)

Resolving data files:   0%|          | 0/4343 [00:00<?, ?it/s]

Keys: ['json', 'mp3', '__key__', '__url__']
Text:  So. Chloe hat gesagt, ich soll noch unten gehen. Was ich natürlich auch machen werde.
Sample rate: 24000 Hz, length: 193968 samples


In [3]:
print(audio["array"])
print(len(audio["array"]))

[-0.00056784 -0.00072023 -0.00080354 ... -0.00294222 -0.00303379
 -0.00226807]
193968


In [4]:
import torch

# transform audio to tensor and add dimension
audio_tensor = torch.tensor(audio["array"]).unsqueeze(0).float()
sr = audio["sampling_rate"]
print(f"Audio tensor: {audio_tensor.shape}, sr={sr}")

Audio tensor: torch.Size([1, 193968]), sr=24000


In [5]:
from TTS.api import TTS

# bypass coqui aggrement
os.environ["COQUI_TOS_AGREED"] = "1"
# Downloads and caches to ~/.local/share/tts/ on first run
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
model = tts.synthesizer.tts_model
print("Model loaded:", type(model).__name__)

C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\jsonlines\jsonlines.py:324: SyntaxWarning: invalid escape sequence '\*'
  :param \*\*kwargs: additional arguments, forwarded to the reader or writer
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Model loaded: Xtts


In [6]:
gpt_cond_latent = model.get_gpt_cond_latents(
    audio_tensor, sr, length=model.config.gpt_cond_len
)
speaker_embedding = model.get_speaker_embedding(audio_tensor, sr)

print("gpt_cond_latent shape:", gpt_cond_latent.shape)
print("speaker_embedding shape:", speaker_embedding.shape)

gpt_cond_latent shape: torch.Size([1, 32, 1024])
speaker_embedding shape: torch.Size([1, 512, 1])


In [7]:
print("--- gpt_cond_latent ---")
print(f"  dtype:  {gpt_cond_latent.dtype}")
print(f"  min:    {gpt_cond_latent.min().item():.4f}")
print(f"  max:    {gpt_cond_latent.max().item():.4f}")
print(f"  mean:   {gpt_cond_latent.mean().item():.4f}")

print("\n--- speaker_embedding ---")
print(f"  dtype:  {speaker_embedding.dtype}")
print(f"  shape:  {speaker_embedding.shape}")
print(
    f"  l2norm: {torch.norm(speaker_embedding).item():.4f}"
)  # should be ~1.0 (L2-normalised)

--- gpt_cond_latent ---
  dtype:  torch.float32
  min:    -13.6700
  max:    14.7195
  mean:   -0.0064

--- speaker_embedding ---
  dtype:  torch.float32
  shape:  torch.Size([1, 512, 1])
  l2norm: 1.0000


In [8]:
print(speaker_embedding)

tensor([[[ 2.6426e-02],
         [-1.2575e-02],
         [ 2.1376e-02],
         [ 2.5637e-02],
         [-9.1736e-02],
         [-2.2620e-02],
         [-3.8188e-03],
         [-5.6407e-02],
         [-2.2110e-02],
         [-7.8064e-03],
         [-4.5025e-02],
         [ 6.7062e-03],
         [-4.8602e-02],
         [ 6.4672e-03],
         [-1.3780e-02],
         [ 2.9092e-03],
         [-5.0539e-02],
         [-9.0480e-02],
         [ 1.6016e-04],
         [ 4.6498e-02],
         [ 2.0797e-02],
         [-5.3174e-02],
         [-8.4345e-03],
         [-2.2500e-02],
         [ 3.7739e-02],
         [-7.6416e-02],
         [-4.9480e-02],
         [-2.0297e-02],
         [-2.5404e-02],
         [-2.1129e-02],
         [-2.2465e-02],
         [ 9.6296e-03],
         [ 1.2642e-02],
         [ 6.9839e-03],
         [-3.1256e-02],
         [-4.6353e-03],
         [-6.5256e-03],
         [ 3.2390e-02],
         [-3.7296e-03],
         [-3.5158e-02],
         [ 1.7186e-02],
         [-3.819

In [9]:
out = model.inference(
    text="Hello, this is a test of voice cloning using the XTTS model.",
    language="en",
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    temperature=model.config.temperature,
    length_penalty=model.config.length_penalty,
    repetition_penalty=model.config.repetition_penalty,
    top_k=model.config.top_k,
    top_p=model.config.top_p,
)

print(f"Synthesized audio length: {len(out['wav'])} samples")
print(f"Duration: {len(out['wav']) / 24000:.2f}s")

Synthesized audio length: 177152 samples
Duration: 7.38s


In [10]:
from IPython.display import Audio, display, HTML

display(HTML("<h3>Original sample</h3>"))
display(Audio(audio["array"], rate=audio["sampling_rate"]))

display(HTML("<h3>Synthesized (cloned voice)</h3>"))
display(Audio(out["wav"], rate=24000))